In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML
from antares_client.search import get_by_id, get_thumbnails
from io import BytesIO
from PIL import Image
import numpy as np
import requests

In [ ]:
def flip_image_bytes(blob):
    """Flip image vertically (ANTARES thumbnails are upside down)."""
    img = Image.open(BytesIO(blob))
    img = img.transpose(Image.FLIP_TOP_BOTTOM)
    buf = BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()

def get_decals_jpg(ra, dec, size=100, layer="ls-dr10", pixscale=0.262):
    """Fetch DECaLS DR10 colour JPG cutout."""
    url = (f"https://www.legacysurvey.org/viewer/cutout.jpg"
           f"?ra={ra}&dec={dec}&layer={layer}&pixscale={pixscale}&size={size}&zoom=16")
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.content

def nJy_to_AB(flux_nJy):
    """Convert flux in nJy to AB magnitude."""
    if flux_nJy is None or flux_nJy <= 0:
        return None
    return -2.5 * np.log10(flux_nJy) + 31.4

def fmt_sigma(val, err):
    """Format value ± error with sigma = |val/err|."""
    if val is None or err is None or err == 0:
        return '—'
    sigma = abs(val / err)
    return f'{val:.2f} ± {err:.2f} ({sigma:.1f}σ)'

# --- UI ---
locus_id_input = widgets.Text(
    placeholder='Enter ANTARES locus ID',
    description='Locus ID:',
    style={'description_width': 'initial'}
)
fetch_btn = widgets.Button(description='Fetch')
output = widgets.Output()

def on_fetch(btn):
    output.clear_output()
    with output:
        locus_id = locus_id_input.value.strip()
        if not locus_id:
            print('Please enter a locus ID.')
            return
        try:
            locus = get_by_id(locus_id)
        except Exception as e:
            print(f'Error fetching locus: {e}')
            return

        # Basic info
        props = locus.properties or {}
        print(f'Locus ID: {locus.locus_id}')
        print(f'RA:       {locus.ra:.4f}')
        print(f'DEC:      {locus.dec:.4f}')

        # Latest alert info
        alerts = locus.alerts or []
        if alerts:
            aprops = alerts[-1].properties or {}
            band = aprops.get("ant_passband") or aprops.get("passband") or "?"

            # Brightest/newest from locus properties (diaSource PSF flux = diff image)
            brightest = props.get("brightest_alert_magnitude")
            newest = props.get("newest_alert_magnitude")
            print(f'\nDiff mag (brightest): {brightest:.2f} ({band})' if brightest else '\nDiff mag (brightest): —')
            print(f'Diff mag (newest):   {newest:.2f} ({band})' if newest else 'Diff mag (newest):   —')

            # Science & template magnitudes from latest alert
            sci_flux = aprops.get("lsst_diaSource_scienceFlux")
            tmpl_flux = aprops.get("lsst_diaSource_templateFlux")
            diff_flux = aprops.get("lsst_diaSource_psfFlux")
            sci_mag = nJy_to_AB(sci_flux)
            tmpl_mag = nJy_to_AB(tmpl_flux)
            diff_mag = nJy_to_AB(diff_flux)
            print(f'Science mag:         {sci_mag:.2f} ({band})  ({sci_flux:.2e} nJy)' if sci_mag else f'Science mag:         —')
            print(f'Template mag:        {tmpl_mag:.2f} ({band})  ({tmpl_flux:.2e} nJy)' if tmpl_mag else f'Template mag:        —')
            print(f'Diff mag (latest):   {diff_mag:.2f} ({band})  ({diff_flux:.2e} nJy)' if diff_mag else f'Diff mag (latest):   —')

        # Catalog checks
        cats = list(locus.catalogs or [])
        print(f'\nCatalogs ({len(cats)}): {cats}')
        for c in ["gaia_dr3_variability", "gaia_dr3_gaia_source", "bright_guide_star_cat"]:
            print(f'  {c}: {"YES" if c in cats else "no"}')

        # Gaia proper motion, parallax, and G mag
        if "gaia_dr3_gaia_source" in cats:
            gaia_rows = locus.catalog_objects.get("gaia_dr3_gaia_source", [])
            for i, row in enumerate(gaia_rows):
                g_mag = row.get("phot_g_mean_mag")
                print(f'  Gaia source {i}:')
                print(f'    G mag    = {g_mag:.2f}' if g_mag else '    G mag    = —')
                print(f'    pmra     = {fmt_sigma(row.get("pmra"), row.get("pmra_error"))} mas/yr')
                print(f'    pmdec    = {fmt_sigma(row.get("pmdec"), row.get("pmdec_error"))} mas/yr')
                print(f'    parallax = {fmt_sigma(row.get("parallax"), row.get("parallax_error"))} mas')

        # Images side by side
        img_widgets = []

        # LSST difference thumbnail from the latest alert
        if alerts:
            alert_id = alerts[-1].alert_id
            print(f'\nFetching thumbnails for alert: {alert_id}')
            thumbs = get_thumbnails(alert_id) or {}
            for ttype, t in thumbs.items():
                if "diff" in str(ttype).lower():
                    img_widgets.append(widgets.VBox([
                        widgets.Label('Latest LSST Difference'),
                        widgets.Image(value=flip_image_bytes(t["blob"]),
                                      format='png', width=200, height=200)
                    ]))
                    break
            else:
                print(f'  No difference thumbnail. Types: {list(thumbs.keys())}')
        else:
            print('\nNo alerts on this locus.')

        # DECaLS cutout
        print('Fetching DECaLS cutout...')
        try:
            decals_blob = get_decals_jpg(locus.ra, locus.dec)
            img_widgets.append(widgets.VBox([
                widgets.Label('DECaLS DR10'),
                widgets.Image(value=decals_blob, format='jpeg', width=200, height=200)
            ]))
        except Exception as e:
            print(f'  DECaLS error: {e}')

        if img_widgets:
            display(widgets.HBox(img_widgets))

fetch_btn.on_click(on_fetch)
display(locus_id_input, fetch_btn, output)